In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install transformers datasets torch scikit-learn -q

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from datasets import load_dataset, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer

In [4]:
model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model_a = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2) # 0: Weak/Noise, 1: Strong Argument

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

In [5]:
print("Fetching IBM Argument Quality Dataset...")
dataset = load_dataset("ibm/argument_quality_ranking_30k", "argument_quality_ranking")
df_ibm = pd.DataFrame(dataset['train'])

Fetching IBM Argument Quality Dataset...


README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

dev.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [6]:
print("Preprocessing and balancing data...")

# Binarize scores
df_ibm['label'] = df_ibm['WA'].apply(lambda x: 1 if x > 0.5 else 0)
df_ibm = df_ibm.rename(columns={'argument': 'text'})[['text', 'label']]

Preprocessing and balancing data...


In [7]:
# ==========================================
# STEP 2 & 3: Load, Polarize, and Balance
# ==========================================
print("Fetching fresh IBM Argument Quality Dataset...")
dataset = load_dataset("ibm/argument_quality_ranking_30k", "argument_quality_ranking")
df_ibm = pd.DataFrame(dataset['train'])

print("Preprocessing and balancing data with Margin of Separation...")

# 1. POLARIZE THE DATA (Remove the grey area)
# The fresh df_ibm has the 'WA' column. We filter it before dropping it.
df_strong = df_ibm[df_ibm['WA'] > 0.6].copy() # Was 0.7
df_strong['label'] = 1
df_weak = df_ibm[df_ibm['WA'] < 0.4].copy()  # Was 0.35
df_weak['label'] = 0

# Combine into a new dataset and rename columns for the Trainer
df_ibm_polarized = pd.concat([df_strong, df_weak])
df_ibm_polarized = df_ibm_polarized.rename(columns={'argument': 'text'})[['text', 'label']]

# Drop nulls and empty strings
df_ibm_polarized = df_ibm_polarized.dropna()
df_ibm_polarized = df_ibm_polarized[df_ibm_polarized['text'].str.strip().astype(bool)]

# 2. FIND THE NEW MINIMUM
count_1 = len(df_ibm_polarized[df_ibm_polarized['label'] == 1])
count_0 = len(df_ibm_polarized[df_ibm_polarized['label'] == 0])
min_count = min(count_1, count_0)

# We want 15% of our 'Weak' data to be courtroom noise
noise_count = int(min_count * 0.15) 
ibm_weak_count = min_count - noise_count

# 3. BALANCE THE DATASET
df_1 = df_ibm_polarized[df_ibm_polarized['label'] == 1].sample(min_count, random_state=42)
df_0_ibm = df_ibm_polarized[df_ibm_polarized['label'] == 0].sample(ibm_weak_count, random_state=42)

# 4. INJECT COURTROOM NOISE
base_noise = [
    "Your Honor.", "Yes, I agree.", "Thank you very much.", 
    "No further questions.", "Good morning to the court.", "I object.", 
    "Please continue.", "Understood.", "That is correct.", "I see.",
    "I challenge that assertion.", "Wait a moment.", "Excuse me."
]
noise_pool = (base_noise * ((noise_count // len(base_noise)) + 1))[:noise_count]
df_0_noise = pd.DataFrame({'text': noise_pool, 'label': 0})

# 5. FINAL SHUFFLE
df_train = pd.concat([df_1, df_0_ibm, df_0_noise]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Polarized Training Data Distribution:\n{df_train['label'].value_counts()}")

Fetching fresh IBM Argument Quality Dataset...
Preprocessing and balancing data with Margin of Separation...
Polarized Training Data Distribution:
label
1    969
0    969
Name: count, dtype: int64


In [8]:
print("Tokenizing data...")
train_ds = Dataset.from_pandas(df_train).map(
    lambda x: tokenizer(x["text"], padding="max_length", truncation=True, max_length=128), batched=True
).train_test_split(test_size=0.1, seed=42) # 10% held out for strict validation

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

# ==========================================
# STEP 5: Train & Save Model A
# ==========================================
training_args = TrainingArguments(
    output_dir="./results_model_a",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    
    # Strategy Alignment
    eval_strategy="epoch",        # Check accuracy every epoch
    save_strategy="epoch",        # Save model every epoch (MUST MATCH EVAL)
    logging_strategy="epoch",
    
    learning_rate=1e-5,
    weight_decay=0.1,             # Stronger regularization
    warmup_steps=50,              # Replaces warmup_ratio
    
    load_best_model_at_end=True,  # Now this will work
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer_a = Trainer(
    model=model_a,
    args=training_args,
    train_dataset=train_ds["train"],
    eval_dataset=train_ds["test"],
    compute_metrics=compute_metrics
)

print("\n--- Starting Final Training for Model A (Evidence Expert) ---")
trainer_a.train()

# Final Save
model_a.save_pretrained("/kaggle/working/LAWNOVA_EVIDENCE_EXPERT")
tokenizer.save_pretrained("/kaggle/working/LAWNOVA_EVIDENCE_EXPERT")

Tokenizing data...


Map:   0%|          | 0/1938 [00:00<?, ? examples/s]


--- Starting Final Training for Model A (Evidence Expert) ---


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,1.414029,1.348676,0.597938
2,1.308668,1.271308,0.649485
3,1.198858,1.152490,0.721649


...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/LAWNOVA_EVIDENCE_EXPERT/tokenizer_config.json',
 '/kaggle/working/LAWNOVA_EVIDENCE_EXPERT/tokenizer.json')

In [9]:
import shutil

# This zips the entire model folder into a file named 'ModelA_Evidence_Expert.zip'
shutil.make_archive('ModelA_Evidence_Expert', 'zip', '/kaggle/working/LAWNOVA_EVIDENCE_EXPERT')

print("Model zipped and ready for download!")

Model zipped and ready for download!


In [10]:
from IPython.display import FileLink
import shutil

# Make sure it's zipped
shutil.make_archive('LAWNOVA_MODEL_A', 'zip', '/kaggle/working/LAWNOVA_EVIDENCE_EXPERT')

# Generate the link
FileLink(r'LAWNOVA_MODEL_A.zip')

/kaggle/working/LAWNOVA_MODEL_A.zip